In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Feature Analysis and Engineering\n",
    "\n",
    "Deep dive into feature importance, correlation analysis, and engineering new features.\n",
    "\n",
    "## Objectives\n",
    "1. Analyze existing features\n",
    "2. Identify most important features\n",
    "3. Create new technical indicators\n",
    "4. Test feature combinations\n",
    "5. Optimize feature set"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.dirname(os.getcwd()))\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy import stats\n",
    "from sklearn.feature_selection import mutual_info_regression, SelectKBest, f_regression\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "from utils.config import DATA_PROCESSED_DIR, STOCK_SYMBOL\n",
    "\n",
    "plt.style.use('seaborn-v0_8')\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Processed Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "filepath = os.path.join(DATA_PROCESSED_DIR, f\"{STOCK_SYMBOL}_processed.csv\")\n",
    "df = pd.read_csv(filepath)\n",
    "df['Date'] = pd.to_datetime(df['Date'])\n",
    "\n",
    "print(f\"Data shape: {df.shape}\")\n",
    "print(f\"\\nColumns: {list(df.columns)}\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Feature Distribution Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select numeric features\n",
    "numeric_cols = df.select_dtypes(include=[np.number]).columns\n",
    "feature_cols = [col for col in numeric_cols if col not in ['target', 'target_class']]\n",
    "\n",
    "print(f\"Total features: {len(feature_cols)}\")\n",
    "print(f\"\\nFeatures: {feature_cols[:10]}...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution plots for key features\n",
    "key_features = ['Close', 'Volume', 'RSI', 'MACD', 'volatility']\n",
    "existing_features = [f for f in key_features if f in df.columns]\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, col in enumerate(existing_features):\n",
    "    axes[i].hist(df[col].dropna(), bins=50, alpha=0.7, edgecolor='black')\n",
    "    axes[i].set_title(f'Distribution of {col}')\n",
    "    axes[i].set_xlabel(col)\n",
    "    axes[i].set_ylabel('Frequency')\n",
    "    axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "# Remove empty subplots\n",
    "for i in range(len(existing_features), len(axes)):\n",
    "    fig.delaxes(axes[i])\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Calculate correlation with target\n",
    "correlations = df[feature_cols].corrwith(df['target']).sort_values(ascending=False)\n",
    "\n",
    "print(\"Top 15 Features Correlated with Target:\")\n",
    "print(correlations.head(15))\n",
    "\n",
    "print(\"\\nBottom 15 Features Correlated with Target:\")\n",
    "print(correlations.tail(15))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot correlation with target\n",
    "plt.figure(figsize=(10, 8))\n",
    "top_corr = correlations.abs().sort_values(ascending=False).head(20)\n",
    "plt.barh(range(len(top_corr)), top_corr.values)\n",
    "plt.yticks(range(len(top_corr)), top_corr.index)\n",
    "plt.xlabel('Absolute Correlation with Target')\n",
    "plt.title('Top 20 Features by Correlation')\n",
    "plt.grid(True, alpha=0.3, axis='x')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature correlation heatmap (top features)\n",
    "top_features = correlations.abs().sort_values(ascending=False).head(15).index\n",
    "corr_matrix = df[top_features].corr()\n",
    "\n",
    "plt.figure(figsize=(12, 10))\n",
    "sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', \n",
    "            center=0, square=True, linewidths=1)\n",
    "plt.title('Correlation Heatmap - Top 15 Features')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Feature Importance (Mutual Information)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Remove rows with NaN in target\n",
    "df_clean = df.dropna(subset=['target'])\n",
    "X = df_clean[feature_cols].fillna(0)\n",
    "y = df_clean['target']\n",
    "\n",
    "# Calculate mutual information\n",
    "mi_scores = mutual_info_regression(X, y, random_state=42)\n",
    "mi_scores = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)\n",
    "\n",
    "print(\"Top 20 Features by Mutual Information:\")\n",
    "print(mi_scores.head(20))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot mutual information scores\n",
    "plt.figure(figsize=(10, 8))\n",
    "top_mi = mi_scores.head(20)\n",
    "plt.barh(range(len(top_mi)), top_mi.values, color='steelblue')\n",
    "plt.yticks(range(len(top_mi)), top_mi.index)\n",
    "plt.xlabel('Mutual Information Score')\n",
    "plt.title('Top 20 Features by Mutual Information')\n",
    "plt.grid(True, alpha=0.3, axis='x')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Feature Scaling Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare before and after scaling\n",
    "sample_features = ['Close', 'Volume', 'RSI']\n",
    "sample_features = [f for f in sample_features if f in df.columns]\n",
    "\n",
    "fig, axes = plt.subplots(len(sample_features), 2, figsize=(12, len(sample_features)*3))\n",
    "\n",
    "scaler = StandardScaler()\n",
    "df_scaled = df[sample_features].fillna(0)\n",
    "df_scaled_transformed = pd.DataFrame(\n",
    "    scaler.fit_transform(df_scaled),\n",
    "    columns=sample_features\n",
    ")\n",
    "\n",
    "for i, col in enumerate(sample_features):\n",
    "    # Before scaling\n",
    "    axes[i, 0].hist(df[col].dropna(), bins=50, alpha=0.7, edgecolor='black')\n",
    "    axes[i, 0].set_title(f'{col} - Original')\n",
    "    axes[i, 0].set_ylabel('Frequency')\n",
    "    \n",
    "    # After scaling\n",
    "    axes[i, 1].hist(df_scaled_transformed[col], bins=50, alpha=0.7, \n",
    "                    edgecolor='black', color='orange')\n",
    "    axes[i, 1].set_title(f'{col} - Scaled')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Creating New Technical Indicators"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Custom indicator: Price Momentum\n",
    "df['momentum_10'] = df['Close'].pct_change(periods=10)\n",
    "df['momentum_20'] = df['Close'].pct_change(periods=20)\n",
    "\n",
    "# Custom indicator: Volume Ratio\n",
    "df['volume_ratio'] = df['Volume'] / df['Volume'].rolling(window=20).mean()\n",
    "\n",
    "# Custom indicator: Price Channel\n",
    "df['price_channel'] = (df['Close'] - df['Low'].rolling(20).min()) / \\\n",
    "                      (df['High'].rolling(20).max() - df['Low'].rolling(20).min())\n",
    "\n",
    "# Custom indicator: Volatility Ratio\n",
    "df['volatility_ratio'] = df['volatility'] / df['volatility'].rolling(20).mean()\n",
    "\n",
    "print(\"New features created:\")\n",
    "print(\"- momentum_10, momentum_20\")\n",
    "print(\"- volume_ratio\")\n",
    "print(\"- price_channel\")\n",
    "print(\"- volatility_ratio\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize new features\n",
    "new_features = ['momentum_10', 'volume_ratio', 'price_channel']\n",
    "\n",
    "fig, axes = plt.subplots(3, 1, figsize=(15, 10))\n",
    "\n",
    "for i, col in enumerate(new_features):\n",
    "    axes[i].plot(df['Date'], df[col], linewidth=1, alpha=0.7)\n",
    "    axes[i].set_title(f'{col} Over Time')\n",
    "    axes[i].set_ylabel(col)\n",
    "    axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "axes[-1].set_xlabel('Date')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Feature Selection Experiments"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select top K features using F-test\n",
    "from sklearn.feature_selection import SelectKBest, f_regression\n",
    "\n",
    "df_clean = df.dropna(subset=['target'])\n",
    "X = df_clean[feature_cols].fillna(0)\n",
    "y = df_clean['target']\n",
    "\n",
    "selector = SelectKBest(score_func=f_regression, k=20)\n",
    "selector.fit(X, y)\n",
    "\n",
    "# Get scores\n",
    "scores = pd.Series(selector.scores_, index=feature_cols).sort_values(ascending=False)\n",
    "\n",
    "print(\"Top 20 Features by F-statistic:\")\n",
    "print(scores.head(20))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare different feature selection methods\n",
    "comparison_df = pd.DataFrame({\n",
    "    'Correlation': correlations.abs().rank(ascending=False),\n",
    "    'Mutual_Info': mi_scores.rank(ascending=False),\n",
    "    'F_Statistic': scores.rank(ascending=False)\n",
    "})\n",
    "\n",
    "comparison_df['Average_Rank'] = comparison_df.mean(axis=1)\n",
    "comparison_df = comparison_df.sort_values('Average_Rank')\n",
    "\n",
    "print(\"\\nTop 15 Features by Average Rank:\")\n",
    "print(comparison_df.head(15))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Feature Interaction Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create interaction features\n",
    "df['RSI_MACD'] = df['RSI'] * df['MACD']\n",
    "df['Volume_Volatility'] = df['Volume'] * df['volatility']\n",
    "df['Price_RSI'] = df['Close'] * df['RSI']\n",
    "\n",
    "interaction_features = ['RSI_MACD', 'Volume_Volatility', 'Price_RSI']\n",
    "\n",
    "# Check correlation with target\n",
    "df_clean = df.dropna(subset=['target'])\n",
    "interaction_corr = df_clean[interaction_features].corrwith(df_clean['target'])\n",
    "\n",
    "print(\"Interaction Feature Correlations with Target:\")\n",
    "print(interaction_corr)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Statistical Tests"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test for normality of features\n",
    "test_features = ['Close', 'Volume', 'RSI', 'price_change_pct']\n",
    "test_features = [f for f in test_features if f in df.columns]\n",
    "\n",
    "print(\"Normality Tests (Shapiro-Wilk):\")\n",
    "print(\"-\" * 50)\n",
    "\n",
    "for feature in test_features:\n",
    "    data = df[feature].dropna()\n",
    "    if len(data) > 0:\n",
    "        stat, p_value = stats.shapiro(data[:5000])  # Sample for speed\n",
    "        is_normal = \"Yes\" if p_value > 0.05 else \"No\"\n",
    "        print(f\"{feature:20s}: p-value = {p_value:.4f} | Normal: {is_normal}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Save Best Features"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select top features based on average rank\n",
    "top_n_features = 20\n",
    "best_features = comparison_df.head(top_n_features).index.tolist()\n",
    "\n",
    "print(f\"\\nTop {top_n_features} Selected Features:\")\n",
    "for i, feature in enumerate(best_features, 1):\n",
    "    print(f\"{i:2d}. {feature}\")\n",
    "\n",
    "# Save to file\n",
    "with open(os.path.join(DATA_PROCESSED_DIR, 'best_features.txt'), 'w') as f:\n",
    "    f.write('\\n'.join(best_features))\n",
    "\n",
    "print(f\"\\n✅ Best features saved to: {DATA_PROCESSED_DIR}/best_features.txt\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusions\n",
    "\n",
    "### Key Findings:\n",
    "1. **Most Important Features**: \n",
    "   - List top 5 features identified\n",
    "\n",
    "2. **Feature Redundancy**:\n",
    "   - Identify highly correlated feature groups\n",
    "\n",
    "3. **New Features**:\n",
    "   - Custom indicators created\n",
    "   - Their effectiveness\n",
    "\n",
    "4. **Recommendations**:\n",
    "   - Which features to keep\n",
    "   - Which to remove\n",
    "   - Ideas for new features"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}